In [10]:
import pandas as pd

# CSV 파일 경로 (파일명 맞게 수정)
file_path = "data/위치관련/서울시 소방서 안전센터 구조대 위치정보.csv"

# CSV 불러오기 (인코딩 문제 대비)
df = pd.read_csv(file_path, encoding="utf-8")

# 1. 연번 컬럼 제거
# 4. 유형구분명 컬럼 제거
# 5. 도서지역포함여부 컬럼 제거 (전부 비어있음)
# 6. 일련번호 컬럼 제거 (전부 비어있음)
cols_to_drop = [
    "연번",
    "유형구분명",
    "도서지역포함여부",
    "일련번호"
]

df = df.drop(columns=cols_to_drop)

# 3. 서ㆍ센터명 컬럼에서 "안전센터" 포함된 row만 남기기
df = df[df["서ㆍ센터명"].str.contains("안전센터", na=False)]

# 결과 확인
print(df.head())

    서ㆍ센터ID       서ㆍ센터명  상위서ㆍ센터ID         X좌표         Y좌표
0  1100106   회현119안전센터   1100000  198522.999  550971.303
6  1102101   동작119안전센터   1102000  192718.705  543910.480
7  1102102  노량진119안전센터   1102000  195310.039  546021.914
8  1102103   상도119안전센터   1102000  193774.330  544394.125
9  1102104   백운119안전센터   1102000  196604.194  543927.680


In [11]:
# X, Y 좌표 범위 확인
print("X 좌표 범위:")
print("min:", df["X좌표"].min())
print("max:", df["X좌표"].max())

print("\nY 좌표 범위:")
print("min:", df["Y좌표"].min())
print("max:", df["Y좌표"].max())

X 좌표 범위:
min: 182618.62
max: 215876.1288

Y 좌표 범위:
min: 538802.5714
max: 565323.0406


EPSG:5186 (Korea 2000 / Central Belt 2010)
흔히 말하는 중부원점 TM 좌표계

## 좌표 변환

In [13]:
from pyproj import Transformer

# 송정119안전센터 제거
df = df[df["서ㆍ센터명"] != "송정119안전센터"]

# 좌표 변환기 생성 (EPSG:5186 → EPSG:4326)
transformer = Transformer.from_crs("EPSG:5186", "EPSG:4326", always_xy=True)

# 변환 수행
lon, lat = transformer.transform(df["X좌표"].values, df["Y좌표"].values)

# 컬럼 추가
df["longitude"] = lon
df["latitude"] = lat

# 인덱스 정리
df = df.reset_index(drop=True)
# 확인
print(df.head())

# 저장
df.to_csv("data/위치관련/서울시 안전센터 위치정보_preprocessed.csv", index=False, encoding="utf-8-sig")

    서ㆍ센터ID       서ㆍ센터명  상위서ㆍ센터ID         X좌표         Y좌표   longitude  \
0  1100106   회현119안전센터   1100000  198522.999  550971.303  126.983284   
1  1102101   동작119안전센터   1102000  192718.705  543910.480  126.917662   
2  1102102  노량진119안전센터   1102000  195310.039  546021.914  126.946952   
3  1102103   상도119안전센터   1102000  193774.330  544394.125  126.929595   
4  1102104   백운119안전센터   1102000  196604.194  543927.680  126.961600   

    latitude  
0  37.558268  
1  37.494623  
2  37.513663  
3  37.498988  
4  37.494800  


C:\Users\DH\AppData\Local\Temp\ipykernel_13876\3177843233.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["longitude"] = lon
C:\Users\DH\AppData\Local\Temp\ipykernel_13876\3177843233.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["latitude"] = lat
